# 03 — Phân loại Email Spam bằng Thống kê Từ khóa (Keyword Statistics)

**Phụ trách:** leehuy (Thành viên 3) — Nhánh `leehuy/statistics-classification`

## Mục tiêu Notebook
1. Huấn luyện bảng trọng số từ khóa (`keyword_stats.json`) **chỉ từ tập `data/train.csv`** bằng tần suất cấp độ email (Document Frequency), làm trơn Laplace và log-ratio chặn trong `[-2.0, +2.0]`.
2. Trực quan hóa **Top 15 từ khóa thiên về Spam (`W > 0`)** và **Top 15 từ khóa thiên về Ham (`W < 0`)** → lưu `outputs/figures/stat_top_keywords.png`.
3. Kiểm chứng công thức tính điểm `score_statistics` bằng `assert` (ví dụ tính tay `1.84`, từ chưa thấy trong train `T05`, tie-break alphabet `T11`, đếm theo cấp độ email).
4. Chấm điểm thống kê `S_stat ∈ [-4.0, +4.0]` trên `data/validation.csv` và `data/test.csv`, vẽ phân bố `stat_score` trên Validation (`outputs/figures/stat_score_distribution.png`).
5. Xuất 2 file chuẩn 5 cột `outputs/stat_val_scores.csv` và `outputs/stat_test_scores.csv` để Trưởng nhóm tích hợp vào mô hình lai Hybrid.

> **Chống Data Leakage:** Tuyệt đối chỉ học trọng số từ `data/train.csv`, không dùng `validation.csv` hay `test.csv` khi đếm tần suất từ. Điểm `S_stat` là điểm phân loại, KHÔNG phải xác suất `% Spam`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import Email
from src.preprocessing import load_emails_from_csv
from src.statistics import (
    get_top_keywords_by_class,
    load_statistics,
    score_statistics,
    save_statistics,
    train_statistics,
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
for rel in ["data/train.csv", "data/validation.csv", "data/test.csv"]:
    assert (PROJECT_ROOT / rel).is_file(), f"Thiếu tệp {rel}. Hãy chạy: python scripts/setup_data.py"
print("Đã sẵn sàng dữ liệu train / validation / test!")

PROJECT_ROOT: D:\uth\Spam_Email
Đã sẵn sàng dữ liệu train / validation / test!


## 1. Huấn luyện bảng trọng số từ khóa từ `data/train.csv`

- Đếm theo **cấp độ email** (`df_spam[w]`, `df_ham[w]`): 1 từ lặp nhiều lần trong cùng 1 email vẫn chỉ tính là 1 email chứa từ đó.
- Chỉ giữ các từ có tổng `df_spam[w] + df_ham[w] >= 3`.
- Công thức làm trơn Laplace & trọng số log-ratio:
  $$P_s(w) = \frac{df_s(w) + 1}{N_s + 2}, \quad P_h(w) = \frac{df_h(w) + 1}{N_h + 2}, \quad W(w) = \text{clip}\left(\ln\frac{P_s(w)}{P_h(w)}, -2.0, +2.0\right)$$

In [2]:
train_emails, train_errors = load_emails_from_csv(
    str(PROJECT_ROOT / "data/train.csv"), require_label=True
)
assert len(train_errors) == 0, f"Có dòng lỗi trong train.csv: {train_errors}"

model = train_statistics(train_emails, min_df=3)
stats_path = save_statistics(model, str(PROJECT_ROOT / "data/processed/keyword_stats.json"))

# Kiểm tra đọc lại từ file JSON vừa lưu
reloaded_model = load_statistics(stats_path)
assert reloaded_model["vocab_size"] == model["vocab_size"]

n_pos = sum(1 for w in model["weights"].values() if w > 0)
n_neg = sum(1 for w in model["weights"].values() if w < 0)
n_zero = sum(1 for w in model["weights"].values() if w == 0)
print(f"Đã lưu bảng trọng số tại: {stats_path}")
print(f"Số email train: n_spam={model['n_spam']}, n_ham={model['n_ham']}, min_df={model['min_df']}")
print(f"Kích thước từ vựng đạt ngưỡng: {model['vocab_size']} từ (W>0: {n_pos}, W<0: {n_neg}, W=0: {n_zero})")

Đã lưu bảng trọng số tại: D:\uth\Spam_Email\data\processed\keyword_stats.json
Số email train: n_spam=175, n_ham=175, min_df=3
Kích thước từ vựng đạt ngưỡng: 1876 từ (W>0: 891, W<0: 891, W=0: 94)


## 2. Trực quan hóa Top 15 từ khóa thiên về Spam & Ham

In [3]:
def plot_top_keywords(stats_model: dict, n: int = 15, out_path: Path | None = None) -> None:
    top_spam, top_ham = get_top_keywords_by_class(stats_model, n=n)

    print(f"Top {n} từ khóa Spam:", [(k.token, k.weight) for k in top_spam[:10]])
    print(f"Top {n} từ khóa Ham :", [(k.token, k.weight) for k in top_ham[:10]])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].barh([k.token for k in top_spam][::-1], [k.weight for k in top_spam][::-1], color="crimson")
    axes[0].set_title(f"Top {n} từ khóa thiên về SPAM (W > 0)")
    axes[0].set_xlabel("Trọng số W")
    axes[0].grid(axis="x", linestyle="--", alpha=0.4)

    axes[1].barh([k.token for k in top_ham][::-1], [k.weight for k in top_ham][::-1], color="steelblue")
    axes[1].set_title(f"Top {n} từ khóa thiên về HAM (W < 0)")
    axes[1].set_xlabel("Trọng số W")
    axes[1].grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, dpi=150)
        print(f"Đã lưu biểu đồ: {out_path}")
    plt.close(fig)

plot_top_keywords(model, n=15, out_path=PROJECT_ROOT / "outputs/figures/stat_top_keywords.png")

Top 15 từ khóa Spam: [('101', 2.0), ('2004', 2.0), ('2005', 2.0), ('ali', 2.0), ('biz', 2.0), ('canon', 2.0), ('cheap', 2.0), ('compliance', 2.0), ('creative', 2.0), ('dealer', 2.0)]
Top 15 từ khóa Ham : [('02', -2.0), ('03', -2.0), ('04', -2.0), ('06', -2.0), ('08', -2.0), ('2000', -2.0), ('2001', -2.0), ('713', -2.0), ('actuals', -2.0), ('agreement', -2.0)]
Đã lưu biểu đồ: D:\uth\Spam_Email\outputs\figures\stat_top_keywords.png


## 3. Kiểm chứng công thức `score_statistics` & Các tình huống biên (T05, T11, Đếm theo Email)

In [4]:
# 1. Kiểm chứng ví dụ tính tay trong 02_GIAI_THICH_THUAT_TOAN.md: (2/5) * (2.0 + 1.5 + 1.1) = 1.84
# Đồng thời kiểm tra T05: từ chưa thấy trong train ('unseen_word') có trọng số 0 và không lỗi
mock_model = {"weights": {"a": 2.0, "b": 1.5, "c": 1.1}}
s_val, ev = score_statistics(["a", "b", "c", "unseen_word"], mock_model, top_k=5)
assert s_val == 1.84, f"Sai công thức: {s_val} != 1.84"
assert len(ev) == 3

# 2. Kiểm chứng T11: khi hòa |W|, sắp xếp phụ theo thứ tự bảng chữ cái để ổn định 100%
tie_model = {"weights": {k: 1.0 for k in ["zeta", "alpha", "beta", "gamma", "delta", "epsilon"]}}
_, ev_tie = score_statistics(list(tie_model["weights"].keys()), tie_model, top_k=5)
assert [k.token for k in ev_tie] == ["alpha", "beta", "delta", "epsilon", "gamma"]

# 3. Kiểm chứng từ lặp trong 1 email chỉ tính 1 lần ở cả train và predict
s_dup, _ = score_statistics(["a", "a", "a", "b", "c"], mock_model, top_k=5)
assert s_dup == 1.84

# 4. Kiểm chứng tính tái lập (T14)
s_run1, ev1 = score_statistics(["free", "money", "enron", "meeting"], model, top_k=5)
s_run2, ev2 = score_statistics(["free", "money", "enron", "meeting"], model, top_k=5)
assert s_run1 == s_run2 and ev1 == ev2

print("PASS toàn bộ kiểm chứng công thức (1.84, T05, T11, deduplicate, T14)!")

PASS toàn bộ kiểm chứng công thức (1.84, T05, T11, deduplicate, T14)!


## 4. Chấm điểm `S_stat` trên `validation.csv` & `test.csv`, trực quan hóa và xuất CSV

Xuất đúng 2 file chuẩn 5 cột `email_id,label,label_num,stat_score,top_keywords`:
- `outputs/stat_val_scores.csv`
- `outputs/stat_test_scores.csv`

In [5]:
def score_stat_dataframe(df: pd.DataFrame, stats_model: dict) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        tokens = str(row["stat_text"]).split() if pd.notna(row["stat_text"]) else []
        stat_score, top_kw = score_statistics(tokens, stats_model, top_k=5)
        kw_str = "; ".join(f"{k.token}({k.weight:+.2f})" for k in top_kw)
        rows.append({
            "email_id": row["email_id"],
            "label": row["label"],
            "label_num": int(row["label_num"]),
            "stat_score": stat_score,
            "top_keywords": kw_str,
        })
    return pd.DataFrame(rows)

val_df = pd.read_csv(PROJECT_ROOT / "data/validation.csv")
test_df = pd.read_csv(PROJECT_ROOT / "data/test.csv")

stat_val = score_stat_dataframe(val_df, model)
stat_test = score_stat_dataframe(test_df, model)

val_out = PROJECT_ROOT / "outputs/stat_val_scores.csv"
test_out = PROJECT_ROOT / "outputs/stat_test_scores.csv"
stat_val.to_csv(val_out, index=False, encoding="utf-8")
stat_test.to_csv(test_out, index=False, encoding="utf-8")

print(f"Đã xuất {val_out.name}: {len(stat_val)} dòng | min={stat_val['stat_score'].min():.4f}, max={stat_val['stat_score'].max():.4f}")
print(f"Đã xuất {test_out.name}: {len(stat_test)} dòng | min={stat_test['stat_score'].min():.4f}, max={stat_test['stat_score'].max():.4f}")

# Thống kê điểm trung bình & trung vị trên Validation theo từng lớp
for lbl in ["spam", "ham"]:
    sub = stat_val[stat_val["label"] == lbl]["stat_score"]
    print(f"Validation [{lbl:4s}] -> count={len(sub)}, mean={sub.mean():+.4f}, median={sub.median():+.4f}, min={sub.min():+.4f}, max={sub.max():+.4f}")

# Vẽ phân bố stat_score trên Validation
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(stat_val[stat_val["label"] == "ham"]["stat_score"], bins=16, alpha=0.65, label="Ham", color="steelblue")
ax.hist(stat_val[stat_val["label"] == "spam"]["stat_score"], bins=16, alpha=0.65, label="Spam", color="crimson")
ax.set_title("Phân bố điểm thống kê S_stat trên tập Validation (75 email)")
ax.set_xlabel("Điểm thống kê S_stat ∈ [-4.0, +4.0]")
ax.set_ylabel("Số lượng email")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
dist_path = PROJECT_ROOT / "outputs/figures/stat_score_distribution.png"
dist_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(dist_path, dpi=150)
plt.close(fig)
print(f"Đã lưu biểu đồ phân bố điểm: {dist_path}")

Đã xuất stat_val_scores.csv: 75 dòng | min=-4.0000, max=4.0000
Đã xuất stat_test_scores.csv: 75 dòng | min=-4.0000, max=4.0000
Validation [spam] -> count=38, mean=+2.0319, median=+2.4000, min=-2.0805, max=+4.0000
Validation [ham ] -> count=37, mean=-3.3511, median=-4.0000, min=-4.0000, max=+2.1306
Đã lưu biểu đồ phân bố điểm: D:\uth\Spam_Email\outputs\figures\stat_score_distribution.png
